# PPO from scratch (3) on Colab

[Open in Colab](https://colab.research.google.com/github/zhizhen-kyle-luo/Robotics_Manipulation/blob/kevin/ppo-start-states/colab/ppo_colab.ipynb)

Runs PPO from scratch on `PegInsertionSide-v1` from Sut's grasped start states: a smoke test, a short tuning run, then the 75M-step runs (dense reward on seeds 1–3, and a one-seed sparse-reward sanity check). Everything goes to Google Drive, so a disconnect costs at most one checkpoint interval (~0.8M steps).

**Before running:** Runtime → Change runtime type → **GPU**. An A100 or L4 is much faster than a T4.

**After a disconnect:** rerun sections 1 and 2 (setup), then rerun the cell that was running. It resumes from its newest checkpoint and skips runs that are already finished.

**Long-running cells (start these early):**

| Cell | Steps | Time |
|---|---|---|
| 4. Tuning | 2 × 10M | 20M / SPS |
| 6. Main runs | 4 × 75M (seed 1 continues the tuning run, so 65M more for it) | ~290M / SPS |
| 7. Final evaluation | 500 states × 4 offsets per run | ~minutes per run |

Section 3 prints SPS and turns it into hours.

## 1. Settings and Google Drive

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/6.4210_peg_ppo"  # runs, checkpoints, logs and results are kept here
BRANCH = "kevin/ppo-start-states"
REPO = "/content/Robotics_Manipulation"

from google.colab import drive
drive.mount("/content/drive")

## 2. Setup (~3 min)

SAPIEN needs a Vulkan driver even for state-only simulation, and Colab ships the NVIDIA driver without the Vulkan files. These are the files from ManiSkill's installation docs; `libvulkan-dev` and the `pip` line match ManiSkill's own Colab quickstart. Colab's preinstalled torch is kept.

If Colab asks to restart the session after `pip`, restart and run sections 1 and 2 again; every step here is safe to repeat.

In [ ]:
import json, os, pathlib

vulkan_files = {
    "/usr/share/vulkan/icd.d/nvidia_icd.json": {
        "file_format_version": "1.0.0",
        "ICD": {"library_path": "libGLX_nvidia.so.0", "api_version": "1.2.155"}},
    "/usr/share/glvnd/egl_vendor.d/10_nvidia.json": {
        "file_format_version": "1.0.0",
        "ICD": {"library_path": "libEGL_nvidia.so.0"}},
    # ManiSkill's docs: needed on some GPUs, such as the A100
    "/etc/vulkan/implicit_layer.d/nvidia_layers.json": {
        "file_format_version": "1.0.0",
        "layer": {
            "name": "VK_LAYER_NV_optimus", "type": "INSTANCE", "library_path": "libGLX_nvidia.so.0",
            "api_version": "1.2.155", "implementation_version": "1", "description": "NVIDIA Optimus layer",
            "functions": {"vkGetInstanceProcAddr": "vk_optimusGetInstanceProcAddr",
                          "vkGetDeviceProcAddr": "vk_optimusGetDeviceProcAddr"},
            "enable_environment": {"__NV_PRIME_RENDER_OFFLOAD": "1"},
            "disable_environment": {"DISABLE_LAYER_NV_OPTIMUS_1": ""}}},
}
for path, content in vulkan_files.items():
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(path).write_text(json.dumps(content, indent=2))

!apt-get -qq update > /dev/null && apt-get -qq install -y --no-install-recommends libvulkan-dev > /dev/null
!pip -q install mani_skill==3.0.1 tyro

if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/zhizhen-kyle-luo/Robotics_Manipulation.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!bash scripts/fetch_start_states.sh

# keep runs, logs and results on Drive so they survive a disconnect
for d in ["runs", "logs", "results"]:
    os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
    if not os.path.islink(d):
        os.symlink(f"{DRIVE_DIR}/{d}", d)

import site
site.main()  # so this kernel sees the packages pip just installed
import ppo_runs

# every PPO run trains on the 1000 train states and is scored on all 500 held-out states
STARTS = ["--starts", "start_states/train/states.pt", "--eval-starts", "start_states/eval/states.pt",
          "--num-eval-envs", "500"]
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

## 3. Smoke test (~15 min)

Read before training anything:
- **hold test**: `grasped` should be close to 1. Sut made the grasps on CPU physics; training runs on GPU physics. If they don't survive, stop: the states need regenerating on the GPU.
- **SPS** from `gpu_smoke.py` is pure simulation; the estimate below uses PPO's own SPS.

In [ ]:
!python scripts/gpu_smoke.py --starts start_states/eval/states.pt --num-envs 1024

The classical controller on the GPU backend, first 100 held-out states. On CPU physics (all 500) it scored `success_once` 0.990 / 0.992 / 0.976 / 0.968 at 0 / 2 / 4 / 6 mm. Numbers far below that mean the GPU controller behaves differently, which also matters for the Diffusion Policy base.

In [ ]:
!python evaluate.py --policy classical --starts start_states/eval/states.pt --limit 100 --num-envs 100 \
    --sim-backend physx_cuda --noise-mm 0 2 4 6 --out results/classical_gpu100.json

PPO at full scale (2048 train envs, 500 eval envs) for 4 iterations, then resumed to 6. The second call has to say `resuming from runs/smoke_ppo/final_ckpt.pt`.

In [ ]:
import re, shutil
shutil.rmtree("runs/smoke_ppo", ignore_errors=True)
ppo_runs.run_ppo("smoke_ppo", STARTS + ["--eval-freq", "2"], 131072, sps_every=1)
ppo_runs.run_ppo("smoke_ppo", STARTS + ["--eval-freq", "2"], 196608, sps_every=1)

sps = [int(x) for x in re.findall(r"^SPS: (\d+)", open("logs/smoke_ppo.log").read(), re.M)]
print(f"\nPPO SPS {sps[-1]:,} (pessimistic: the smoke run evaluates every 2 iterations, the real runs every 25)")
for label, steps in [("tuning, 2 x 10M", 20e6), ("main runs, ~290M", 290e6)]:
    print(f"  {label}: {steps / sps[-1] / 3600:.1f} h")

## 4. Tuning (long: 2 × 10M steps)

ManiSkill's PPO baseline was tuned for joint-space control; here every method uses end-effector deltas, where the baseline's initial noise (log std −0.5) means about 6 cm per step. This compares it with less noise. Both use dense reward and seed 1, and the winner's run is reused as the first 10M steps of its 75M seed-1 run.

Forcing the gripper closed (`--gripper-closed`) is left out: on CPU, PPO learned to keep the peg within 100k steps by itself. Add `"closed": ["--gripper-closed"]` to `CONFIGS` to test it anyway.

In [ ]:
TUNE_STEPS = 10_000_000
CONFIGS = {"base": [], "lowstd": ["--logstd-init", "-1.5"]}

def run_name(config, reward, seed):
    return f"ppo_{config}_{'dense' if reward == 'normalized_dense' else 'sparse'}_seed{seed}"

for config, flags in CONFIGS.items():
    ppo_runs.run_ppo(run_name(config, "normalized_dense", 1),
                     STARTS + ["--seed", "1", "--reward-mode", "normalized_dense"] + flags, TUNE_STEPS)

## 5. Compare and choose

Pick the config with the higher `eval success_once`. If both are still at 0 (likely this early: the ManiSkill paper reports PPO struggling on this task), pick the higher `eval return` and `train/return`. Set `CHOSEN` in the next section.

In [ ]:
!python scripts/curves.py runs/ppo_*_dense_seed1
%load_ext tensorboard
%tensorboard --logdir runs

## 6. Main runs (long: 75M steps each)

Each cell is one run. Rerunning a cell resumes it, and a finished run is skipped. Seed 1 continues its tuning run. If Colab time runs short, the proposal's cut order is to drop to fewer PPO seeds, so run these top to bottom.

In [ ]:
CHOSEN = "base"  # from section 5
TOTAL = 75_000_000

def main_run(reward, seed):
    ppo_runs.run_ppo(run_name(CHOSEN, reward, seed),
                     STARTS + ["--seed", str(seed), "--reward-mode", reward] + CONFIGS[CHOSEN], TOTAL)

In [ ]:
main_run("normalized_dense", 1)

In [ ]:
main_run("normalized_dense", 2)

In [ ]:
main_run("normalized_dense", 3)

In [ ]:
main_run("sparse", 1)  # Kyle's sanity check: one seed with sparse reward

## 7. Final evaluation and learning curves

Scores each finished run's final checkpoint on all 500 held-out states at 0 / 2 / 4 / 6 mm, and the classical controller on the same GPU backend so the comparison is like for like. Results and the learning-curve CSV land in `results/` on Drive.

In [ ]:
for reward, seed in [("normalized_dense", 1), ("normalized_dense", 2), ("normalized_dense", 3), ("sparse", 1)]:
    name = run_name(CHOSEN, reward, seed)
    if not ppo_runs.finished(f"runs/{name}", TOTAL):
        print(f"{name}: not finished, skipped")
        continue
    kwargs = json.dumps({"checkpoint": f"runs/{name}/final_ckpt.pt"})
    print(name)
    !python evaluate.py --policy ppo_policy:make_policy --policy-kwargs '{kwargs}' --starts start_states/eval/states.pt \
        --num-envs 100 --sim-backend physx_cuda --out results/{name}.json

print("classical")
!python evaluate.py --policy classical --starts start_states/eval/states.pt --num-envs 100 --sim-backend physx_cuda \
    --out results/classical_gpu500.json
!python scripts/curves.py runs/ppo_{CHOSEN}_* --csv results/ppo_curves.csv